# Parte 3 — Extensión: búsqueda híbrida BM25 + densa con RRF (Opción B)

**Nombres: Jessica Ballesteros, Miguel Álvarez, Darlyn Ludeña**

## Declaración previa (escrita antes de medir)

**Lo que midió el baseline (Parte 2, k=5):** Hit Rate 0,625 y MRR 0,542 sobre 8 respondibles, con abstención correcta de 1,0. Fallan tres respondibles:

- **Caso 2 (DPO):** documento equivocado; los cinco vecinos son de InstructGPT.
- **Casos 3 (CoT) y 4 (InstructGPT):** documento correcto, pero el fragmento 0000, que tiene el resumen con la frase esperada, no entra en el top-5.

**Modo de falla que esperamos corregir:** fragmentos que el **vector** no pone arriba pero que comparten **términos exactos** con la pregunta. Las preguntas del golden set están en español, pero varias citan nombres propios que solo aparecen en la cabecera del fragmento 0000 de cada paper: *Rafailov* (caso 2), *Ouyang* (caso 4), *Bai* (caso 5), *Vaswani* (caso 1). BM25 los encuentra por coincidencia exacta y RRF los sube en la fusión.

**Predicción:** debería corregir los **casos 2 y 4**, y mejorar el MRR del caso 5 (hoy en la posición 3). **No** debería corregir el **caso 3**: la pregunta no nombra a Wei et al. ni usa un término distintivo del resumen.

**Riesgo declarado:** fuera de los nombres propios y siglas (RLHF, GPT-3, DPO), la pregunta en español casi no comparte vocabulario con documentos en inglés. BM25 aporta poco y puede meter ruido que empeore preguntas que hoy aciertan. Además, parte de la ganancia vendría de que **nuestras** preguntas citan autores; un usuario real no siempre lo haría. Si la extensión no mejora, es un resultado válido y lo explicamos.

## Configuración

Es la misma del baseline: mismo corpus, fragmentación 512/102, bge-m3 en la H200, Qdrant en el contenedor (en una colección propia) y el mismo generador. Lo único que cambia es `retrieve` (ver `hibrido.py`).

In [1]:
from pathlib import Path
import os, sys, json
import pandas as pd

raiz = Path.cwd()
while raiz != raiz.parent and not ((raiz / "pyproject.toml").exists() and (raiz / "rag_pipeline.py").exists()):
    raiz = raiz.parent
sys.path.insert(0, str(raiz))
os.chdir(raiz)

os.environ["EMBEDDING_BACKEND"] = "h200"
os.environ["QDRANT_URL"] = "http://localhost:6333"
os.environ["QDRANT_COLLECTION"] = "mmia6013_rag_hibrido"
os.environ["OPENAI_API_KEY"] = ""
os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["OLLAMA_URL"] = "http://172.28.230.10:11434"
CHUNK_TOKENS, OVERLAP_TOKENS = 512, 102

import rag_pipeline
from rag_pipeline import EMBEDDING_MODEL
from evaluation import load_golden_set, evaluate_retrieval, escribir_csv
from hibrido import PipelineHibrido, SoloDenso, RRF_K, CANDIDATOS

assert rag_pipeline.QDRANT_URL != ":memory:"
assert "172.28.230.10" in rag_pipeline.OLLAMA_URL
golden = load_golden_set(raiz / "golden_set.json")
carpeta = raiz / "resultados" / "parte3"
carpeta.mkdir(parents=True, exist_ok=True)
print("RRF_K =", RRF_K, "| candidatos por lista =", CANDIDATOS, "| preguntas =", len(golden))

RRF_K = 60 | candidatos por lista = 20 | preguntas = 10


## Método

Para cada pregunta:

1. **Denso:** los 20 fragmentos más cercanos en Qdrant (bge-m3), como el baseline.
2. **BM25:** los 20 fragmentos con mayor puntaje BM25 sobre **los mismos 530 fragmentos**, con una tokenización simple: minúsculas, sin tildes, solo alfanuméricos. Los fragmentos con BM25 = 0 no cuentan como candidatos.
3. **RRF:** `score(d) = Σ 1 / (60 + posición de d en cada lista)`. Se usan solo las posiciones, no los puntajes, porque el coseno (entre 0 y 1) y BM25 (sin tope) no son comparables.
4. Se entregan los top-k de la fusión al mismo prompt y al mismo generador.

In [2]:
pipeline = PipelineHibrido()
chunks = pipeline.ingest(raiz / "corpus", chunk_tokens=CHUNK_TOKENS, overlap_tokens=OVERLAP_TOKENS)
pipeline.index(chunks)   # indexa en Qdrant y construye BM25 sobre los mismos fragmentos
print("Fragmentos:", len(chunks), "| vectores en Qdrant:", pipeline.client.count(pipeline.collection, exact=True).count)

/Users/miguelestebanalvarez/Posgrado Maestría Inteligencia Artificial/1er Semestre/IA Generativa y Agentes/Talleres/Taller 02/taller-02-rag-alvarez-ballesteros-ludeña/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (30809 > 8192). Running this sequence through the model will result in indexing errors


ingesta: bai-2022-constitutional-ai.pdf: 34 página(s), 94294 caracteres útiles → 75 fragmento(s)
ingesta: brown-2020-gpt3.pdf: 75 página(s), 186342 caracteres útiles → 156 fragmento(s)
ingesta: ouyang-2022-instructgpt.pdf: 68 página(s), 143167 caracteres útiles → 118 fragmento(s)
ingesta: rafailov-2023-dpo.pdf: 27 página(s), 71700 caracteres útiles → 66 fragmento(s)
ingesta: vaswani-2017-attention-is-all-you-need.pdf: 15 página(s), 31566 caracteres útiles → 25 fragmento(s)
ingesta: wei-2022-chain-of-thought.pdf: 43 página(s), 105503 caracteres útiles → 90 fragmento(s)
Fragmentos: 530 | vectores en Qdrant: 530


## 3.1 — Recuperación: baseline vs. híbrido (sin generar, costo cero)

Se evalúan los dos sistemas **sobre el mismo índice**. El baseline se recalcula aquí con `SoloDenso`, que usa solo la lista densa; sus cifras deben coincidir con las de la Parte 2.

In [3]:
filas_tabla = []
reportes_ret = {}
for k in (3, 5):
    for nombre, sistema in (("baseline (denso)", SoloDenso(pipeline)), ("híbrido BM25+RRF", pipeline)):
        rep = evaluate_retrieval(golden, sistema, k=k, generar=False)
        reportes_ret[(nombre, k)] = rep
        filas_tabla.append({"sistema": nombre, "k": k, "hit_rate": rep["hit_rate"], "mrr": rep["mrr"]})

df_ret = pd.DataFrame(filas_tabla)
display(df_ret.round(3))

# Control: el baseline recalculado debe coincidir con la Parte 2
p2 = pd.read_csv(raiz / "resultados" / "parte2_k5.csv")
p2_resp = p2[p2["respondible"]]
base5 = df_ret[(df_ret.sistema == "baseline (denso)") & (df_ret.k == 5)].iloc[0]
print(f"Control k=5 → Parte 2: hit={p2_resp['hit'].astype(bool).mean():.3f} mrr={p2_resp['reciprocal_rank'].mean():.3f} "
      f"| aquí: hit={base5.hit_rate:.3f} mrr={base5.mrr:.3f}")

,sistema,k,hit_rate,mrr
0,baseline (denso),3,0.625,0.542
1,híbrido BM25+RRF,3,0.625,0.562
2,baseline (denso),5,0.625,0.542
3,híbrido BM25+RRF,5,0.625,0.562


Control k=5 → Parte 2: hit=0.625 mrr=0.542 | aquí: hit=0.625 mrr=0.542


In [4]:
# Detalle por pregunta (k=5): posición del primer acierto en cada sistema, y de qué lista vino
detalle = []
for q in golden:
    if not q["documentos_fuente"]:
        continue
    fila_b = next(r for r in reportes_ret[("baseline (denso)", 5)]["rows"] if r["id"] == q["id"])
    fila_h = next(r for r in reportes_ret[("híbrido BM25+RRF", 5)]["rows"] if r["id"] == q["id"])
    hits_h = pipeline.retrieve(q["pregunta"], top_k=5)
    primero = hits_h[fila_h["posicion"] - 1] if fila_h["posicion"] else None
    terminos_bm25 = [h["chunk_id"] for h in pipeline.retrieve_bm25(q["pregunta"], 3)]
    detalle.append({
        "id": q["id"], "tipo": q["tipo"],
        "pos_baseline": fila_b["posicion"], "pos_hibrido": fila_h["posicion"],
        "rank_denso_del_acierto": primero["rank_denso"] if primero else None,
        "rank_bm25_del_acierto": primero["rank_bm25"] if primero else None,
        "top3_bm25": " ".join(terminos_bm25),
    })
df_detalle = pd.DataFrame(detalle)
display(df_detalle)
df_detalle.to_csv(carpeta / "parte3_detalle_por_pregunta.csv", index=False, encoding="utf-8-sig")

,id,tipo,pos_baseline,pos_hibrido,rank_denso_del_acierto,rank_bm25_del_acierto,top3_bm25
0,1,simple,1.0,1.0,1.0,12.0,brown-2020-gpt3-0123 ouyang-2022-instructgpt-0...
1,2,simple,NaN,NaN,NaN,NaN,brown-2020-gpt3-0123 brown-2020-gpt3-0122 ouya...
2,3,simple,NaN,NaN,NaN,NaN,ouyang-2022-instructgpt-0106 brown-2020-gpt3-0...
3,4,simple,NaN,NaN,NaN,NaN,brown-2020-gpt3-0123 ouyang-2022-instructgpt-0...
4,5,simple,3.0,2.0,3.0,8.0,brown-2020-gpt3-0123 ouyang-2022-instructgpt-0...
5,6,multi-chunk,1.0,1.0,1.0,NaN,brown-2020-gpt3-0123 ouyang-2022-instructgpt-0...
6,7,multi-chunk,1.0,1.0,1.0,NaN,ouyang-2022-instructgpt-0106 ouyang-2022-instr...
7,10,adversarial,1.0,1.0,1.0,6.0,brown-2020-gpt3-0123 ouyang-2022-instructgpt-0...


## 3.2 — Abstención con el híbrido (genera: 10 preguntas × 2 valores de k)

Mismo prompt, mismo generador y misma frase de abstención que el baseline. Solo cambia qué fragmentos llegan al contexto.

In [5]:
filas_h = {}
for k in (3, 5):
    rep = evaluate_retrieval(golden, pipeline, k=k, generar=True)
    generadores = {r["generador"] for r in rep["rows"]}
    assert all(g and g.startswith("ollama:") for g in generadores), f"Generador inesperado: {generadores}"
    escribir_csv(rep["rows"], carpeta / f"parte3_hibrido_k{k}.csv", f"{EMBEDDING_MODEL}+bm25-rrf")
    filas_h[k] = rep["rows"]
    print(f"k={k}: CSV escrito ({len(rep['rows'])} filas) · generador {generadores}")

escribir_csv([*filas_h[3], *filas_h[5]], carpeta / "resultados_parte3.csv", f"{EMBEDDING_MODEL}+bm25-rrf")

k=3: CSV escrito (10 filas) · generador {'ollama:qwen3:32b'}
k=5: CSV escrito (10 filas) · generador {'ollama:qwen3:32b'}


## Tabla comparativa (derivada de los CSV crudos)

In [6]:
def metricas_desde_csv(ruta, sistema):
    df = pd.read_csv(ruta)
    resp, neg = df[df["respondible"]], df[~df["respondible"]]
    return {"sistema": sistema, "k": int(df["k"].iloc[0]),
            "hit_rate": resp["hit"].astype(bool).mean(), "mrr": resp["reciprocal_rank"].mean(),
            "abstencion_correcta": neg["abstuvo"].astype(bool).mean(),
            "abstencion_indebida": resp["abstuvo"].astype(bool).mean(),
            "csv": str(Path(ruta).relative_to(raiz))}

tabla = pd.DataFrame(
    [metricas_desde_csv(raiz / "resultados" / f"parte2_k{k}.csv", "baseline (denso)") for k in (3, 5)] +
    [metricas_desde_csv(carpeta / f"parte3_hibrido_k{k}.csv", "híbrido BM25+RRF") for k in (3, 5)]
).sort_values(["k", "sistema"])
display(tabla.round(3))
tabla.to_csv(carpeta / "parte3_resumen.csv", index=False, encoding="utf-8-sig")

for k in (3, 5):
    b = tabla[(tabla.k == k) & (tabla.sistema.str.startswith("baseline"))].iloc[0]
    h = tabla[(tabla.k == k) & (tabla.sistema.str.startswith("híbrido"))].iloc[0]
    print(f"k={k}: ΔHit Rate = {h.hit_rate - b.hit_rate:+.3f} · ΔMRR = {h.mrr - b.mrr:+.3f} · "
          f"Δabst. indebida = {h.abstencion_indebida - b.abstencion_indebida:+.3f}")

,sistema,k,hit_rate,mrr,abstencion_correcta,abstencion_indebida,csv
0,baseline (denso),3,0.625,0.542,1.0,0.125,resultados/parte2_k3.csv
2,híbrido BM25+RRF,3,0.625,0.562,1.0,0.125,resultados/parte3/parte3_hibrido_k3.csv
1,baseline (denso),5,0.625,0.542,1.0,0.250,resultados/parte2_k5.csv
3,híbrido BM25+RRF,5,0.625,0.562,1.0,0.250,resultados/parte3/parte3_hibrido_k5.csv


k=3: ΔHit Rate = +0.000 · ΔMRR = +0.021 · Δabst. indebida = +0.000
k=5: ΔHit Rate = +0.000 · ΔMRR = +0.021 · Δabst. indebida = +0.000


## Diagnóstico del resultado (v1)

El híbrido **no** corrigió los casos 2 y 4, contra lo que predijimos. Para ver por qué, se inspecciona qué fragmentos pone BM25 arriba y qué términos de la pregunta comparten con ellos.

In [7]:
from hibrido import tokenizar
from evaluation import acierta

for q in golden:
    if q["id"] not in (2, 3, 4):
        continue
    terminos = pipeline.terminos_consulta(q["pregunta"])
    puntajes = pipeline._bm25.get_scores(terminos)
    orden = list(pd.Series(puntajes).sort_values(ascending=False).index)
    objetivo = next(i for i, c in enumerate(pipeline._chunks)
                    if acierta({"document": c.document, "text": c.text}, q))
    print("=" * 100)
    print(f"Caso {q['id']}: términos de la consulta = {terminos}")
    print(f"Fragmento esperado {pipeline._chunks[objetivo].id}: posición BM25 = {orden.index(objetivo) + 1}, "
          f"puntaje = {puntajes[objetivo]:.2f}, términos que comparte = "
          f"{sorted(set(terminos) & set(tokenizar(pipeline._chunks[objetivo].text)))}")
    for i in orden[:3]:
        c = pipeline._chunks[i]
        comunes = sorted(set(terminos) & set(tokenizar(c.text)))
        print(f"  top BM25: {c.id}  puntaje={puntajes[i]:.2f}  términos comunes={comunes}")
        print(f"      «{c.text[:160]}…»")

Caso 2: términos de la consulta = ['que', 'metodo', 'de', 'alineamiento', 'propone', 'el', 'articulo', 'de', 'rafailov', 'et', 'al', 'y', 'que', 'objetivo', 'de', 'entrenamiento', 'usa']
Fragmento esperado rafailov-2023-dpo-0000: posición BM25 = 35, puntaje = 8.46, términos que comparte = ['rafailov']
  top BM25: brown-2020-gpt3-0123  puntaje=35.49  términos comunes=['al', 'de', 'que']
      «instars than females. = Target Completion → L’analyse de la distribution de fr ́ equence des stades larvaires d’I. verticalis dans une s ́ erie d’ ́ etangs a ́ …»
  top BM25: brown-2020-gpt3-0122  puntaje=29.59  términos comunes=['de', 'que']
      «artist? A: Target Completion → MARCEL DUCHAMP Target Completion → r mutt Target Completion → duchamp Target Completion → marcel duchamp Target Completion → R.Mu…»
  top BM25: ouyang-2022-instructgpt-0106  puntaje=29.46  términos comunes=['de', 'et', 'que']
      «set, along with model samples and the human-written demonstration. These 5 prompts were se

## Interpretación (v1)

**Resultado.** El Hit Rate no cambia (0,625 con k=3 y con k=5). El MRR sube de 0,542 a 0,563 (+0,021), y ese cambio viene de **una sola pregunta**: el caso 5 (Constitutional AI) pasa de la posición 3 a la 2, porque su fragmento esperado estaba 3.º en la lista densa y 8.º en BM25, y RRF suma ambas. La abstención no cambia (correcta 1,0; indebida 0,125 con k=3 y 0,25 con k=5).

**Predicción frente a resultado.**

| Caso | Predicción | Resultado |
|---|---|---|
| 2 (DPO, cita a *Rafailov*) | se corrige | ❌ no se corrige |
| 4 (InstructGPT, cita a *Ouyang*) | se corrige | ❌ no se corrige |
| 5 (CAI, cita a *Bai*) | mejora el MRR | ✅ posición 3 → 2 |
| 3 (CoT, sin autor) | no se corrige | ✅ no se corrige |

**Por qué falló (celda de diagnóstico).** Para casi todas las preguntas, los primeros puestos de BM25 son los **mismos** fragmentos: pasajes en francés y en español de los apéndices de GPT-3 (traducción) y de InstructGPT (ejemplos multilingües). Las palabras vacías de nuestras preguntas ("que", "de", "la", "se") casi no aparecen en un corpus en inglés; BM25 les da un IDF alto, y en esos pasajes pesan más que el nombre del autor. En el caso 2, el resumen de DPO solo comparte "rafailov" con la pregunta y queda por debajo de esos pasajes.

Es el riesgo que declaramos antes de medir (preguntas en español contra documentos en inglés), pero por un mecanismo que no previmos: el problema no es que BM25 "no encuentre nada", sino que encuentra con fuerza **lo equivocado**.

## 3.3 — Iteración declarada: v2, sin palabras vacías del español en la consulta

**Cambio (uno solo):** se quitan de la **pregunta** las palabras vacías del español (y "et", de "et al.") antes de calcular BM25. El índice BM25, los fragmentos, la lista densa, RRF, el prompt y el generador son **los mismos** que en la v1; la v2 reutiliza el índice de la v1 (`compartir_indice`), sin volver a vectorizar.

**Predicción, escrita antes de medir:**

- **Caso 2:** se corrige. Sin palabras vacías, el término más discriminante es "rafailov", que solo aparece en el fragmento del resumen de DPO; ese fragmento ya estaba 9.º en la lista densa, y RRF debería meterlo en el top-5.
- **Caso 4:** probablemente **no** se corrige. "ouyang" aparece en las referencias de muchos fragmentos de otros papers, así que no discrimina.
- **Caso 3:** no se corrige: el fragmento esperado está 21.º en la lista densa, fuera de los 20 candidatos.
- **Riesgo:** con menos términos, BM25 puede traer fragmentos de referencias bibliográficas que citan a los autores, y empeorar preguntas que hoy aciertan.

La v1 **se reporta igual**: la comparación es baseline → v1 → v2 sobre el mismo golden set.

In [8]:
from hibrido import STOPWORDS_ES

hibrido_v2 = PipelineHibrido(stopwords=STOPWORDS_ES)
hibrido_v2.compartir_indice(pipeline)     # mismo Qdrant y mismo BM25 que la v1
print("Colección:", hibrido_v2.collection, "| fragmentos:", len(hibrido_v2._chunks))
for q in golden:
    if q["id"] in (2, 4):
        print(f"Caso {q['id']}: v1 → {pipeline.terminos_consulta(q['pregunta'])}")
        print(f"        v2 → {hibrido_v2.terminos_consulta(q['pregunta'])}")

Colección: mmia6013_rag_hibrido | fragmentos: 530
Caso 2: v1 → ['que', 'metodo', 'de', 'alineamiento', 'propone', 'el', 'articulo', 'de', 'rafailov', 'et', 'al', 'y', 'que', 'objetivo', 'de', 'entrenamiento', 'usa']
        v2 → ['metodo', 'alineamiento', 'propone', 'articulo', 'rafailov', 'objetivo', 'entrenamiento', 'usa']
Caso 4: v1 → ['que', 'tecnica', 'de', 'alineamiento', 'presentan', 'ouyang', 'et', 'al', 'para', 'que', 'los', 'modelos', 'sigan', 'mejor', 'la', 'intencion', 'del', 'usuario']
        v2 → ['tecnica', 'alineamiento', 'presentan', 'ouyang', 'modelos', 'sigan', 'mejor', 'intencion', 'usuario']


In [9]:
# Recuperación (sin generar) de los tres sistemas sobre el mismo índice
filas_ret = []
reportes_v2 = {}
for k in (3, 5):
    for nombre, sistema in (("baseline (denso)", SoloDenso(pipeline)),
                            ("híbrido v1", pipeline),
                            ("híbrido v2 (sin stopwords)", hibrido_v2)):
        rep = evaluate_retrieval(golden, sistema, k=k, generar=False)
        reportes_v2[(nombre, k)] = rep
        filas_ret.append({"sistema": nombre, "k": k, "hit_rate": rep["hit_rate"], "mrr": rep["mrr"]})
display(pd.DataFrame(filas_ret).round(3))

# Detalle por pregunta (k=5)
det = []
for q in golden:
    if not q["documentos_fuente"]:
        continue
    pos = {n: next(r["posicion"] for r in reportes_v2[(n, 5)]["rows"] if r["id"] == q["id"])
           for n in ("baseline (denso)", "híbrido v1", "híbrido v2 (sin stopwords)")}
    det.append({"id": q["id"], "tipo": q["tipo"], **{f"pos_{n}": p for n, p in pos.items()}})
df_det_v2 = pd.DataFrame(det)
display(df_det_v2)
df_det_v2.to_csv(carpeta / "parte3_v2_detalle_por_pregunta.csv", index=False, encoding="utf-8-sig")

,sistema,k,hit_rate,mrr
0,baseline (denso),3,0.625,0.542
1,híbrido v1,3,0.625,0.562
2,híbrido v2 (sin stopwords),3,0.750,0.688
3,baseline (denso),5,0.625,0.542
4,híbrido v1,5,0.625,0.562
5,híbrido v2 (sin stopwords),5,0.750,0.688


,id,tipo,pos_baseline (denso),pos_híbrido v1,pos_híbrido v2 (sin stopwords)
0,1,simple,1.0,1.0,1.0
1,2,simple,NaN,NaN,1.0
2,3,simple,NaN,NaN,NaN
3,4,simple,NaN,NaN,NaN
4,5,simple,3.0,2.0,2.0
5,6,multi-chunk,1.0,1.0,1.0
6,7,multi-chunk,1.0,1.0,1.0
7,10,adversarial,1.0,1.0,1.0


In [10]:
# Abstención con la v2 (genera: 10 preguntas × 2 valores de k)
filas_v2 = {}
for k in (3, 5):
    rep = evaluate_retrieval(golden, hibrido_v2, k=k, generar=True)
    generadores = {r["generador"] for r in rep["rows"]}
    assert all(g and g.startswith("ollama:") for g in generadores), f"Generador inesperado: {generadores}"
    escribir_csv(rep["rows"], carpeta / f"parte3_hibrido_v2_k{k}.csv", f"{EMBEDDING_MODEL}+bm25-rrf-v2")
    filas_v2[k] = rep["rows"]
    print(f"k={k}: CSV escrito · generador {generadores}")
escribir_csv([*filas_v2[3], *filas_v2[5]], carpeta / "resultados_parte3_v2.csv", f"{EMBEDDING_MODEL}+bm25-rrf-v2")

k=3: CSV escrito · generador {'ollama:qwen3:32b'}
k=5: CSV escrito · generador {'ollama:qwen3:32b'}


In [11]:
# Tabla final de la Parte 3, derivada de los CSV crudos
tabla_final = pd.DataFrame(
    [metricas_desde_csv(raiz / "resultados" / f"parte2_k{k}.csv", "baseline (denso)") for k in (3, 5)] +
    [metricas_desde_csv(carpeta / f"parte3_hibrido_k{k}.csv", "híbrido v1") for k in (3, 5)] +
    [metricas_desde_csv(carpeta / f"parte3_hibrido_v2_k{k}.csv", "híbrido v2 (sin stopwords)") for k in (3, 5)]
).sort_values(["k", "sistema"])
display(tabla_final.round(3))
tabla_final.to_csv(carpeta / "parte3_resumen_final.csv", index=False, encoding="utf-8-sig")

,sistema,k,hit_rate,mrr,abstencion_correcta,abstencion_indebida,csv
0,baseline (denso),3,0.625,0.542,1.0,0.125,resultados/parte2_k3.csv
2,híbrido v1,3,0.625,0.562,1.0,0.125,resultados/parte3/parte3_hibrido_k3.csv
4,híbrido v2 (sin stopwords),3,0.750,0.688,1.0,0.250,resultados/parte3/parte3_hibrido_v2_k3.csv
1,baseline (denso),5,0.625,0.542,1.0,0.250,resultados/parte2_k5.csv
3,híbrido v1,5,0.625,0.562,1.0,0.250,resultados/parte3/parte3_hibrido_k5.csv
5,híbrido v2 (sin stopwords),5,0.750,0.688,1.0,0.250,resultados/parte3/parte3_hibrido_v2_k5.csv


## Interpretación (v2)

*(Se completa con las cifras de esta corrida: predicción frente a resultado para los casos 2, 3 y 4, y si la v2 empeoró alguna pregunta que la v1 acertaba.)*